# Artificial Neural Networks and Deep Learning


## ⚙️ Import Libraries

In [ ]:
import os
from datetime import datetime
from PIL import Image as im

import numpy as np
import pandas as pd
import random

import tensorflow as tf
from tensorflow import keras as tfk
from tensorflow.keras import layers as tfkl

import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)
tf.random.set_seed(42)

seed=42

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tfk.__version__}")
print(f"GPU devices: {len(tf.config.list_physical_devices('GPU'))}")

## ⏳ Load the Data

In [ ]:
data = np.load("mars_for_students.npz")

training_set = data["training_set"]
X_train = training_set[:, 0]
y_train = training_set[:, 1]

X_test = data["test_set"]

print(f"Training X shape: {X_train.shape}")
print(f"Training y shape: {y_train.shape}")
print(f"Test X shape: {X_test.shape}")

In [ ]:
SPLITS_SIZE=500

In [ ]:
#splitting the test set into validation and test set
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=SPLITS_SIZE, random_state=seed
)

print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

# Alien Stermination

In [ ]:
#clean dataset
invalid_inxs=[]
reference_label=y_train[62].copy() #it's an image with an alien inside
for i in range(len(X_train)):
    if np.array_equal(y_train[i], reference_label):
        invalid_inxs.append(i)

X_train = np.delete(X_train, invalid_inxs, axis=0)
y_train = np.delete(y_train, invalid_inxs, axis=0)

print(X_train.shape)
print(y_train.shape)

# Metrics Definition

In [ ]:
from tensorflow.keras.saving import register_keras_serializable

In [ ]:
# Visualization callback to show solutions after some epochs
class VizCallback(tf.keras.callbacks.Callback):
    def __init__(self, image_path, label_path, frequency=5):
        super().__init__()
        self.image_path = image_path
        self.label_path = label_path
        self.frequency = frequency

    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.frequency == 0:  # Visualize only every "frequency" epochs
            image, label = load_single_image(self.image_path, self.label_path)
            label = apply_category_mapping(label)
            image = tf.expand_dims(image, 0)
            pred = self.model.predict(image, verbose=0)
            y_pred = tf.math.argmax(pred, axis=-1)
            y_pred = y_pred.numpy()

            # Create colormap
            num_classes = 5
            colormap = create_segmentation_colormap(num_classes)

            plt.figure(figsize=(16, 4))

            # Input image
            plt.subplot(1, 3, 1)
            plt.imshow(image[0])
            plt.title("Input Image")
            plt.axis('off')

            # Ground truth
            plt.subplot(1, 3, 2)
            colored_label = apply_colormap(label.numpy(), colormap)
            plt.imshow(colored_label)
            plt.title("Ground Truth Mask")
            plt.axis('off')

            # Prediction
            plt.subplot(1, 3, 3)
            colored_pred = apply_colormap(y_pred[0], colormap)
            plt.imshow(colored_pred)
            plt.title("Predicted Mask")
            plt.axis('off')

            plt.tight_layout()
            plt.show()
            plt.close()


# Define custom Mean Intersection Over Union metric to find intersection btw predicted and actual labels
@register_keras_serializable()
class MeanIntersectionOverUnion(tf.keras.metrics.MeanIoU): #override of the original class called this way
    def __init__(self, num_classes, labels_to_exclude=None, name="mean_iou", dtype=None, **kwargs):
        # Use ignore_class if provided, otherwise use labels_to_exclude
        ignore_class = kwargs.pop('ignore_class', labels_to_exclude)

        super(MeanIntersectionOverUnion, self).__init__(num_classes=num_classes, name=name, dtype=dtype)
        if labels_to_exclude is None:
            labels_to_exclude = [0]  # Default to excluding label 0 from final network evaluation (not from training?)
        self.labels_to_exclude = labels_to_exclude

    def update_state(self, y_true, y_pred, sample_weight=None):
        # Convert predictions to class labels
        y_pred = tf.math.argmax(y_pred, axis=-1)

        # Flatten the tensors
        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.reshape(y_pred, [-1])

        # Apply mask to exclude specified labels
        for label in self.labels_to_exclude:
            mask = tf.not_equal(y_true, label)
            y_true = tf.boolean_mask(y_true, mask)
            y_pred = tf.boolean_mask(y_pred, mask)

        # Update the state
        return super().update_state(y_true, y_pred, sample_weight)

def load_single_image(image_path, label_path, input_size=(64, 128)):
    """
    Load a single image-label pair with the correct shape.
    """
    image = X_val[image_path]
    label = y_val[label_path]

    # Read and preprocess the image
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    image = image[..., np.newaxis]/255.0
    image = tf.image.resize(image, input_size)   # Resize to fixed size

    # Read and preprocess the label
    label = tf.convert_to_tensor(label, dtype=tf.int32) # Convert to Tensor
    label = tf.expand_dims(label, axis=-1) # Add channel dimension
    label = tf.image.resize(label, input_size, method='nearest') # Resize label
    label = tf.squeeze(label, axis=-1) # Remove channel dimension
    label = tf.cast(label, dtype=tf.int32)

    return image, label

category_map = {
        0: 0,  # backgroung
        1: 1,  # soil
        2: 2,  # bedrock
        3: 3,  # sand
        4: 4,  # big rock
}

def apply_category_mapping(label):
    """
    Apply category mapping to labels.
    """
    keys_tensor = tf.constant(list(category_map.keys()), dtype=tf.int32)
    vals_tensor = tf.constant(list(category_map.values()), dtype=tf.int32)
    table = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(keys_tensor, vals_tensor),
        default_value=0
    )
    return table.lookup(label)

def create_segmentation_colormap(num_classes):
    """
    Create a linear colormap using a predefined palette.
    Uses 'viridis' as default because it is perceptually uniform
    and works well for colorblindness.
    """
    return plt.cm.viridis(np.linspace(0, 1, num_classes))

def apply_colormap(label, colormap=None):
    """
    Apply the colormap to a label.
    """
    # Ensure label is 2D
    label = np.squeeze(label)

    if colormap is None:
        num_classes = len(np.unique(label))
        colormap = create_segmentation_colormap(num_classes)

    # Apply the colormap
    colored = colormap[label.astype(int)]

    return colored

def plot_sample_batch(dataset, num_samples=3):
    """
    Display some image and label pairs from the dataset.
    """
    plt.figure(figsize=(15, 4*num_samples))

    for images, labels in dataset.take(1):
        labels_np = labels.numpy()
        num_classes = len(np.unique(labels_np))
        colormap = create_segmentation_colormap(num_classes)

        for j in range(min(num_samples, len(images))):
            # Plot original image
            plt.subplot(num_samples, 2, j*2 + 1)
            plt.imshow(images[j])
            plt.title(f'Image {j+1}')
            plt.axis('off')

            # Plot colored label
            plt.subplot(num_samples, 2, j*2 + 2)
            colored_label = apply_colormap(labels_np[j], colormap)
            plt.imshow(colored_label)
            plt.title(f'Label {j+1}')
            plt.axis('off')

    plt.tight_layout()
    plt.show()
    plt.close()


In [ ]:
# Dice Coefficient function
@register_keras_serializable()
def dice_coefficient(y_true, y_pred, num_classes=3, epsilon=1e-6):
    # Convert logits to predicted class labels (if necessary)
    y_pred = tf.argmax(y_pred, axis=-1)  # Convert to class indices

    # Ensure y_true is integer type for class indices (cast if necessary)
    y_true = tf.cast(y_true, tf.int32)

    # Create one-hot encoded version of y_true and y_pred for Dice calculation
    y_true_one_hot = tf.one_hot(y_true, num_classes)
    y_pred_one_hot = tf.one_hot(y_pred, num_classes)

    # Flatten the tensors
    y_true_f = tf.reshape(y_true_one_hot, [-1])
    y_pred_f = tf.reshape(y_pred_one_hot, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)

    return (2. * intersection + epsilon) / (union + epsilon)

# Dice Loss function
@register_keras_serializable()
def dice_loss(y_true, y_pred, num_classes=3):
     return 1 - dice_coefficient(y_true, y_pred, num_classes)

@register_keras_serializable()
def combined_loss(y_true, y_pred, num_classes=3):
    ce_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)(y_true, y_pred)
    dice_loss_value = dice_loss(y_true, y_pred, num_classes)
    return ce_loss + dice_loss_value

# Augmentation function definition

In [ ]:
def heavy_augmentation(images):
    augmented_images = []

    for i in range(images.shape[0]):
        image = tf.convert_to_tensor(images[i], dtype=tf.float32)
        image = tf.expand_dims(image, axis=-1)  # Add channel dimension
        delta = tf.random.uniform([], 0.20, 0.5)
        contrast_factor = tf.random.uniform([], 0.20, 0.5)
        image = tf.image.adjust_brightness(image, delta)
        image = tf.image.adjust_contrast(image, contrast_factor)

        augmented_images.append(image[..., 0])

    return tf.stack(augmented_images)

In [ ]:
def shuffle_inplace(images, labels, seed=None):
    # Convert images and labels to tf.Variable for in-place updates
    images = tf.Variable(images)
    labels = tf.Variable(labels)

    num_samples = images.shape[0]

    # Generate random indices
    random_indices = np.random.permutation(num_samples)

    # Shuffle images and labels using TensorFlow's tf.gather
    images.assign(tf.gather(images, random_indices))  # Use assign() for in-place update
    labels.assign(tf.gather(labels, random_indices))  # Same here for labels

    return images, labels

# Creating Datasets

In [ ]:
BATCH_SIZE = 32

In [ ]:
def make_dataset(images, labels, batch_size, shuffle=None):
    """
    Create a memory-efficient TensorFlow dataset.
    """
    # Create dataset from file paths
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=batch_size * 2, seed=seed)

    # Batch the data
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
X2_train = X_train.copy()
y2_train = y_train.copy()
X2_train = heavy_augmentation(X2_train)

print(X2_train.shape)
print(y2_train.shape)


In [ ]:
X2_train, y2_train = shuffle_inplace(images=X2_train, labels=y2_train, seed=seed)

In [ ]:
#merging
X3_train = np.concatenate((X_train, X2_train), axis=0)
y3_train = np.concatenate((y_train, y2_train), axis=0)

print(X3_train.shape)
print(y3_train.shape)

In [ ]:
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

In [ ]:
train_dataset = make_dataset(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataset = make_dataset(
    X_val, y_val,
    batch_size=BATCH_SIZE,
)
# Check the shape of the data
for images, labels in train_dataset.take(1):
    input_shape = images.shape[1:]
    num_classes = len(np.unique(labels))
    print(f"\nInput shape: {input_shape}")
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    print(f"Number of classes: {num_classes}")
    break

# 🛠️ Train and Save the Model

In [ ]:
# Set learning rate for the optimiser
LEARNING_RATE = 0.001

# Set early stopping patience threshold
PATIENCE = 40

# Set maximum number of training epochs
EPOCHS = 1000

batch_size = 32

In [ ]:
from tensorflow.keras.regularizers import l2

In [ ]:
def unet_block(input_tensor, filters, kernel_size=3, activation='relu', stack=2, name=''):
    # Initialise the input tensor
    x = input_tensor

    # Apply a sequence of Conv2D, Batch Normalisation, and Activation layers for the specified number of stacks
    for i in range(stack):
        x = tfkl.Conv2D(filters, kernel_size=kernel_size, kernel_regularizer=l2(0.01), padding='same', name=name + 'conv' + str(i + 1))(x)
        x = tfkl.BatchNormalization(name=name + 'bn' + str(i + 1))(x)
        x = tfkl.Activation(activation, name=name + 'activation' + str(i + 1))(x)

    # Return the transformed tensor
    return x

In [ ]:
from tensorflow.keras.saving import register_keras_serializable
@register_keras_serializable()
class AugmentationLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AugmentationLayer, self).__init__(**kwargs)

    def call(self, inputs, training=False):
        # Apply augmentations only during training
        if training:
            # Random horizontal flip (50% chance)
            if random.random() > 0.5:
                inputs = tf.image.flip_left_right(inputs)

            # Apply random brightness and contrast augmentations
            inputs = tf.image.random_brightness(inputs, max_delta=0.2)  # Random brightness
            inputs = tf.image.random_contrast(inputs, lower=0.8, upper=1.2)  # Random contrast

        return inputs

In [ ]:
def aspp_block(input_tensor, num_filters, name="aspp"):
    """Atrous Spatial Pyramid Pooling block"""
    # 1x1 Convolution (No dilation)
    conv1x1 = tf.keras.layers.Conv2D(num_filters, 1, padding='same', activation='relu', name=f"{name}_conv1x1")(input_tensor)

    # 3x3 Convolutions with different dilation rates
    conv3x3_r1 = tf.keras.layers.Conv2D(num_filters, 3, dilation_rate=1, padding='same', activation='relu', name=f"{name}_conv3x3_r1")(input_tensor)
    conv3x3_r6 = tf.keras.layers.Conv2D(num_filters, 3, dilation_rate=6, padding='same', activation='relu', name=f"{name}_conv3x3_r6")(input_tensor)
    conv3x3_r12 = tf.keras.layers.Conv2D(num_filters, 3, dilation_rate=12, padding='same', activation='relu', name=f"{name}_conv3x3_r12")(input_tensor)

    # Global Average Pooling
    global_avg_pool = tf.keras.layers.GlobalAveragePooling2D(name=f"{name}_global_avg_pool")(input_tensor)
    global_avg_pool = tf.keras.layers.Reshape((1, 1, input_tensor.shape[-1]), name=f"{name}_reshape")(global_avg_pool)
    global_avg_pool = tf.keras.layers.Conv2D(num_filters, 1, padding='same', activation='relu', name=f"{name}_global_conv")(global_avg_pool)
    global_avg_pool = tf.keras.layers.UpSampling2D(size=(input_tensor.shape[1], input_tensor.shape[2]), interpolation='bilinear', name=f"{name}_upsample")(global_avg_pool)

    # Concatenate all features
    concatenated = tf.keras.layers.Concatenate(name=f"{name}_concat")([conv1x1, conv3x3_r1, conv3x3_r6, conv3x3_r12, global_avg_pool])

    # Output projection
    output = tf.keras.layers.Conv2D(num_filters, 1, padding='same', activation='relu', name=f"{name}_output")(concatenated)

    return output

In [ ]:
# Define the model
input_layer = tfkl.Input(shape=input_shape, name='input_layer')

augmented_input = AugmentationLayer()(input_layer)

# Downsampling path
down_block_1 = unet_block(augmented_input, 32, name='down_block1_')
d1 = tfkl.MaxPooling2D()(down_block_1)
d1 = tfkl.Dropout(0.3)(d1)

down_block_2 = unet_block(d1, 64, name='down_block2_')
d2 = tfkl.MaxPooling2D()(down_block_2)
d2 = tfkl.Dropout(0.3)(d2)

down_block_3 = unet_block(d2, 128, name='down_block3_')
d3 = tfkl.MaxPooling2D()(down_block_3)
d3 = tfkl.Dropout(0.3)(d3)

down_block_4 = unet_block(d3, 256, name='down_block4_')
d4 = tfkl.MaxPooling2D()(down_block_4)
d4 = tfkl.Dropout(0.3)(d4)

# New deeper block
down_block_5 = unet_block(d4, 512, name='down_block5_')
d5 = tfkl.MaxPooling2D()(down_block_5)
d5 = tfkl.Dropout(0.3)(d5)

# Bottleneck
bottleneck = aspp_block(d5, 1024, name='aspp_bottleneck')
bottleneck = tfkl.Dropout(0.5)(bottleneck)

# Upsampling path
u1 = tfkl.UpSampling2D()(bottleneck)
u1 = tfkl.Concatenate()([u1, down_block_5])
u1 = unet_block(u1, 512, name='up_block1_')
u1 = tfkl.Dropout(0.3)(u1)

u2 = tfkl.UpSampling2D()(u1)
u2 = tfkl.Concatenate()([u2, down_block_4])
u2 = unet_block(u2, 256, name='up_block2_')
u2 = tfkl.Dropout(0.3)(u2)

u3 = tfkl.UpSampling2D()(u2)
u3 = tfkl.Concatenate()([u3, down_block_3])
u3 = unet_block(u3, 128, name='up_block3_')
u3 = tfkl.Dropout(0.3)(u3)

u4 = tfkl.UpSampling2D()(u3)
u4 = tfkl.Concatenate()([u4, down_block_2])
u4 = unet_block(u4, 64, name='up_block4_')
u4 = tfkl.Dropout(0.3)(u4)

u5 = tfkl.UpSampling2D()(u4)
u5 = tfkl.Concatenate()([u5, down_block_1])
u5 = unet_block(u5, 32, name='up_block5_')
u5 = tfkl.Dropout(0.3)(u5)

# Output layer
output_layer = tfkl.Conv2D(
    num_classes,
    kernel_size=1,
    padding='same',
    activation="softmax",
    name='output_layer'
)(u5)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name='Deeper_UNet')

In [ ]:
# Print the model summary to inspect layer names
model.summary(expand_nested=False, show_trainable=True)

# Generate and display a graphical representation of the model architecture.
tf.keras.utils.plot_model(model, show_trainable=True, expand_nested=True, dpi=70)

In [ ]:
# Compile the model
print("Compiling model...")
model.compile(
    loss=combined_loss,
    optimizer=tf.keras.optimizers.AdamW(LEARNING_RATE, clipvalue=3.0),
    metrics=["accuracy", MeanIntersectionOverUnion(num_classes=5, labels_to_exclude=None), dice_coefficient] #not very relevant in this case, we look at MeanIntersection
)
print("Model compiled!")

In [ ]:
from tensorflow.keras.mixed_precision import set_global_policy

# Set mixed precision policy
set_global_policy('float32')

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [ ]:
# Setup callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=20,
)

lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=10, min_lr=1e-7)

viz_callback = VizCallback(image_path=160, label_path=160, frequency=5)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    batch_size=32,
    epochs=150,
    validation_data=val_dataset,
    callbacks=[early_stopping, viz_callback, lr_scheduler],
    verbose=1,
).history

# Calculate and print the final validation accuracy
final_val_meanIoU = round(max(history['val_mean_iou'])* 100, 2)
print(f'Final validation Mean Intersection Over Union: {final_val_meanIoU}%')

In [ ]:
# Plot and display training and validation loss
plt.figure(figsize=(18, 3))
plt.plot(history['loss'], label='Training', alpha=0.8, color='#ff7f0e', linewidth=2)
plt.plot(history['val_loss'], label='Validation', alpha=0.9, color='#5a9aa5', linewidth=2)
plt.title('Cross Entropy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Plot and display training and validation accuracy
plt.figure(figsize=(18, 3))
plt.plot(history['accuracy'], label='Training', alpha=0.8, color='#ff7f0e', linewidth=2)
plt.plot(history['val_accuracy'], label='Validation', alpha=0.9, color='#5a9aa5', linewidth=2)
plt.title('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Plot and display training and validation mean IoU
plt.figure(figsize=(18, 3))
plt.plot(history['mean_iou'], label='Training', alpha=0.8, color='#ff7f0e', linewidth=2)
plt.plot(history['val_mean_iou'], label='Validation', alpha=0.9, color='#5a9aa5', linewidth=2)
plt.title('Mean Intersection over Union')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Save the trained model to a file with the accuracy included in the filename
model_filename = 'Net_'+str(final_val_meanIoU)+'.keras'
model.save(model_filename)
#del model

print(f"Model saved to {model_filename}")

## 📊 Prepare Your Submission

In our Kaggle competition, submissions are made as `csv` files. To create a proper `csv` file, you need to flatten your predictions and include an `id` column as the first column of your dataframe. To maintain consistency between your results and our solution, please avoid shuffling the test set. The code below demonstrates how to prepare the `csv` file from your model predictions.




In [ ]:
# If model_filename is not defined, load the most recent model from Google Drive
if "model_filename" not in globals() or model_filename is None:
    files = [f for f in os.listdir('.') if os.path.isfile(f) and f.startswith('model_') and f.endswith('.keras')]
    files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    if files:
        model_filename = files[0]
    else:
        raise FileNotFoundError("No model files found in the current directory.")

In [ ]:
custom_objects = {
    'MeanIntersectionOverUnion': MeanIntersectionOverUnion,
    'combined_loss': combined_loss,
    'dice_coefficient': dice_coefficient,
    'dice_loss': dice_loss}
    #'ResizeLayer': ResizeLayer }
model = tfk.models.load_model(model_filename, custom_objects = custom_objects )
print(f"Model loaded from {model_filename}")

In [ ]:
preds = model.predict(X_test)
preds = np.argmax(preds, axis=-1)
print(f"Predictions shape: {preds.shape}")

In [ ]:
class_to_color = {0:[0,0,0],1:[150, 0, 0],2:[0 , 150, 0],3:[0,0,150],4:[150,150,0],5:[50, 0 , 150],6:[0 , 150,150],7:[200,120,30]}
plt.imshow(X_test[1000], cmap='gray')
plt.show()
plt.imshow([[class_to_color[u] for u in uu ] for uu in preds[1000]])
plt.show()
plt.clf()

In [ ]:
def y_to_df(y) -> pd.DataFrame:
    """Converts segmentation predictions into a DataFrame format for Kaggle."""
    n_samples = len(y)
    y_flat = y.reshape(n_samples, -1)
    df = pd.DataFrame(y_flat)
    df["id"] = np.arange(n_samples)
    cols = ["id"] + [col for col in df.columns if col != "id"]
    return df[cols]

In [ ]:
# Create and download the csv submission file
timestep_str = model_filename.replace("model_", "").replace(".keras", "")
submission_filename = f"submission_{timestep_str}.csv"
submission_df = y_to_df(preds)
submission_df.to_csv(submission_filename, index=False)

#  
<img src="https://airlab.deib.polimi.it/wp-content/uploads/2019/07/airlab-logo-new_cropped.png" width="350">

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/9/95/Instagram_logo_2022.svg/800px-Instagram_logo_2022.svg.png" width="15"> **Instagram:** https://www.instagram.com/airlab_polimi/

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/81/LinkedIn_icon.svg/2048px-LinkedIn_icon.svg.png" width="15"> **LinkedIn:** https://www.linkedin.com/company/airlab-polimi/
___
Credits: Alberto Archetti 📧 alberto.archetti@polito.it





```
   Copyright 2024 Alberto Archetti

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```